In [72]:
# set working directory
import os
base_dir = os.getcwd()
base_path = os.path.join(base_dir, "Raw", "SocioeconomicDatasets")

print(f"Current project root: {base_dir}")
print(f"Socioeconomic data path: {base_path}")

Current project root: /Users/yuttingc1/Downloads/qtmCapstonepollution
Socioeconomic data path: /Users/yuttingc1/Downloads/qtmCapstonepollution/Raw/SocioeconomicDatasets


In [ ]:
# install packages if necessary and import packages
# pip install pandas geopandas xarray rasterstats rasterio rioxarray censusdata openpyxl netCDF4

import pandas as pd
import glob
import re
import geopandas as gpd
import zipfile
import xarray as xr
from rasterstats import zonal_stats
from rasterio.transform import from_bounds
from scipy.stats import pearsonr
from functools import reduce
import warnings
warnings.filterwarnings('ignore')

In [12]:
# socioeconomic data cleaning 
def process_acs(folder_name, table_code, compute_variable):
    folder_path = os.path.join(base_path, folder_name)
    files = glob.glob(os.path.join(folder_path, f"ACSDT5Y*.{table_code}-Data.csv"))
    df_list = []
    for file in files:
        year = int(re.search(r"ACSDT5Y(\d{4})", file).group(1))
        df = pd.read_csv(file, skiprows=[1])
        # keep tract level only
        df["GEO_ID"] = df["GEO_ID"].astype(str)
        df = df[df["GEO_ID"].str.startswith("1400000US13")]
        df["tract_fips"] = df["GEO_ID"].str[-11:]
        # convert numeric columns
        for col in df.columns:
            if "E" in col:
                df[col] = pd.to_numeric(df[col], errors="coerce")
        # compute variable
        df = compute_variable(df)
        df["year"] = year
        df_list.append(df[["tract_fips", "value", "year"]])
    final_df = pd.concat(df_list, ignore_index=True)
    return final_df

def education_var(df):
    df["value"] = (
        df["B15003_022E"] +
        df["B15003_023E"] +
        df["B15003_024E"] +
        df["B15003_025E"]
    ) / df["B15003_001E"].replace(0, pd.NA) #percent with bachelor's or higher
    return df
def income_var(df):
    df["value"] = df["B19013_001E"]  # median household income
    return df
def population_var(df):
    df["value"] = df["B01003_001E"] # total population
    return df
def poverty_var(df):
    df["value"] = df["B17001_002E"] / df["B17001_001E"].replace(0, pd.NA) #percent in poverty
    return df
def race_var(df):
    df["value"] = 1 - (df["B02001_002E"] / df["B02001_001E"].replace(0, pd.NA)) #percent non-white
    return df
def unemployment_var(df):
    df["value"] = df["B23025_005E"] / df["B23025_002E"].replace(0, pd.NA) #percent unemployed
    return df

education = process_acs("Education 2010 - 2024", "B15003", education_var)
education = education.rename(columns={"value": "education"})
income = process_acs("Median Income 2010 - 2024", "B19013", income_var)
income = income.rename(columns={"value": "income"})
population = process_acs("Population 2010 - 2024", "B01003", population_var)
population = population.rename(columns={"value": "population"})
poverty = process_acs("Poverty Status 2010 - 2024", "B17001", poverty_var)
poverty = poverty.rename(columns={"value": "poverty_rate"})
race = process_acs("Race 2010 - 2024", "B02001", race_var)
race = race.rename(columns={"value": "minority_share"})
unemployment = process_acs("Unemployment 2010 - 2024", "B23025", unemployment_var)
unemployment = unemployment.rename(columns={"value": "unemployment_rate"})
data = education.copy()

data = data.merge(income, on=["tract_fips", "year"], how="left", validate="one_to_one")
data = data.merge(population, on=["tract_fips", "year"], how="left", validate="one_to_one")
data = data.merge(poverty, on=["tract_fips", "year"], how="left", validate="one_to_one")
data = data.merge(race, on=["tract_fips", "year"], how="left", validate="one_to_one")
data = data.merge(unemployment, on=["tract_fips", "year"], how="left", validate="one_to_one")

data = data.sort_values(["tract_fips", "year"]).reset_index(drop=True)

socioeconomic_panel = data
print(socioeconomic_panel.shape)
print(socioeconomic_panel.head())
print(socioeconomic_panel.isna().sum())

# Impute missing values with mean by year
num_cols = [
    "education",
    "income",
    "population",
    "poverty_rate",
    "minority_share",
    "unemployment_rate"
]

for col in num_cols:
    socioeconomic_panel[col] = socioeconomic_panel[col].fillna(
        socioeconomic_panel.groupby("year")[col].transform("mean")
    )
print(socioeconomic_panel.isna().sum())

/var/folders/06/tq96mkbj073500scjzpzwlrw0000gn/T/ipykernel_53439/2401277712.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["value"] = df["B17001_002E"] / df["B17001_001E"].replace(0, pd.NA) #percent in poverty
/var/folders/06/tq96mkbj073500scjzpzwlrw0000gn/T/ipykernel_53439/2401277712.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["year"] = year
/var/folders/06/tq96mkbj073500scjzpzwlrw0000gn/T/ipykernel_53439/2401277712.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of ca

(29732, 8)
    tract_fips education  year   income  population poverty_rate  \
0  13001950100  0.095168  2012  38807.0        3040     0.073779   
1  13001950100  0.103013  2013  42941.0        3162     0.129269   
2  13001950100  0.106207  2014  47045.0        3403     0.140954   
3  13001950100  0.118821  2015  50338.0        3410     0.106289   
4  13001950100  0.136499  2016  57000.0        3144     0.120833   

  minority_share unemployment_rate  
0       0.064145          0.040278  
1       0.074004          0.062087  
2       0.059947           0.08972  
3       0.029032           0.09648  
4       0.045165          0.108143  
tract_fips             0
education            162
year                   0
income               358
population             0
poverty_rate         184
minority_share       162
unemployment_rate    194
dtype: int64
tract_fips           0
education            0
year                 0
income               0
population           0
poverty_rate         0
minorit

/var/folders/06/tq96mkbj073500scjzpzwlrw0000gn/T/ipykernel_53439/2401277712.py:86: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  socioeconomic_panel[col] = socioeconomic_panel[col].fillna(


In [13]:
# output to csv
socioeconomic_panel.to_csv("Cleaning/socioeconomic_panel.csv",index=False)

In [73]:
# CDC pm2.5 data cleaning
cdc_pm_path = os.path.join(base_dir, "Raw", "CDCPM25")

pm1620 = pd.read_csv(os.path.join(cdc_pm_path, "pm16-20.csv"))
pm1115 = pd.read_csv(os.path.join(cdc_pm_path, "pm11-15.csv"))

print(f"Loaded 2011-2015 data: {pm1115.shape}")
print(f"Loaded 2016-2020 data: {pm1620.shape}")

pm1620.groupby("year")["date"].nunique()
def clean_pm25(df):
    df = df.copy()
    df = df.rename(columns={
        "ctfips": "tract_fips",
        "DS_PM_pred": "pm25"
    })

    df["tract_fips"] = df["tract_fips"].astype(str).str.zfill(11)
    df["pm25"] = pd.to_numeric(df["pm25"], errors="coerce")
    df["date"] = pd.to_datetime(df["date"], format="%d%b%Y", errors="coerce")
    df["year"] = df["date"].dt.year
    df = df.dropna(subset=["tract_fips", "pm25", "year"])
    return df
pm1115_clean = clean_pm25(pm1115)
pm1620_clean = clean_pm25(pm1620)
pm_all = pd.concat([pm1115_clean, pm1620_clean], ignore_index=True)
pm25_yearly = (
    pm_all.groupby(["tract_fips", "year"])["pm25"]
    .agg(
        pm25_mean="mean",
        pm25_median="median",
        pm25_count="count",
        pm25_std="std"
    )
    .reset_index()
)
final_df = socioeconomic_panel.merge(
    pm25_yearly,
    on=["tract_fips", "year"],
    how="inner"
)

Loaded 2011-2015 data: (3586264, 9)
Loaded 2016-2020 data: (3892740, 9)


In [15]:
pm25_yearly.to_csv("Cleaning/pm25_11-20.csv",index=False)

In [ ]:
# urban rural data classification
import os
import pandas as pd

base_dir = os.getcwd()

urban_rural_path = os.path.join(base_dir, "Raw", "Urban Rural Separation")

urban10 = pd.read_excel(os.path.join(urban_rural_path, "18-19 urban rural.xlsx"))
urban20 = pd.read_csv(os.path.join(urban_rural_path, "RUCA-codes-2020-clean.csv"))

print(f"Loaded 2018-2019 Urban/Rural data: {urban10.shape}")
print(f"Loaded 2020 Urban/Rural data: {urban20.shape}")

ruca = urban10.copy()
ruca = ruca[["GEOID", "Primary RUCA Code 2010"]]
ruca = ruca.rename(columns={
    "GEOID": "tract_fips",
    "Primary RUCA Code 2010": "ruca_code"
})

ruca["urban"] = ruca["ruca_code"].apply(
    lambda x: 1 if x == 1 else 0
)
ruca["tract_fips"] = ruca["tract_fips"].astype(str).str.zfill(11)
final_df["tract_fips"] = final_df["tract_fips"].astype(str).str.zfill(11) 

df_1119 = final_df[final_df["year"] <= 2019].copy()
df_2020 = final_df[final_df["year"] == 2020].copy()
df_2020["tract_fips"] = df_2020["tract_fips"].astype(str).str.zfill(11)

df_1119 = df_1119.merge(
    ruca[["tract_fips", "urban"]],
    on="tract_fips",
    how="left",
    validate="many_to_one"
)

urban20["tract_fips"] = urban20["TractFIPS20"].astype(str).str[-11:]
urban20["tract_fips"] = urban20["tract_fips"].astype(str).str.zfill(11)

urban20["urban"] = urban20["UrbanCore"].apply(
    lambda x: 1 if x == 1 else 0
)
urban20 = urban20[["tract_fips", "urban"]]

df_2020 = df_2020.merge(
    urban20[["tract_fips", "urban"]],
    on="tract_fips",
    how="left",
    validate="many_to_one"
)
df_2020["urban"].value_counts(dropna=False)
final_df = pd.concat([df_1119, df_2020], ignore_index=True)
final_df["urban"].value_counts(dropna=False)


urban
1    12308
0     6200
Name: count, dtype: int64

In [19]:
# constructing vulnerability index
df = final_df.copy()

df["education_inv"] = 1 - df["education"] # higher education means less vulnerability
df["income_inv"] = -df["income"] # higher income means less vulnerability, negate to align direction with other variables

# Normalize
from sklearn.preprocessing import StandardScaler

features = [
    "education_inv",
    "income_inv",
    "poverty_rate",
    "minority_share",
    "unemployment_rate"
]

df_scaled = df.copy()

df_scaled[features] = StandardScaler().fit_transform(df_scaled[features])

# Build vulnerability index as average of standardized variables
df["vulnerability_index"] = df_scaled[features].mean(axis=1)

df["Year_Date"] = pd.to_datetime(df["year"], format="%Y")

df_urban = df[df["urban"] == 1].copy()
df_rural = df[df["urban"] == 0].copy()



In [20]:
# constructing fairness index
# Definition: The fairness index measures the extent to which a community experiences 
# more or less pollution than expected given its level of socioeconomic vulnerability.
df["pm25_scaled"] = StandardScaler().fit_transform(df[["pm25_mean"]])

import statsmodels.api as sm
df["interaction"] = df["vulnerability_index"] * df["urban"]

X = df[["vulnerability_index", "urban", "interaction"]]
X = sm.add_constant(X)

model = sm.OLS(df["pm25_mean"], X).fit()

df["fairness_index"] = -model.resid

df["fairness_index"].describe()

df["fairness_group"] = pd.qcut(
    df["fairness_index"],
    4,
    labels=["Low", "Mid-Low", "Mid-High", "High"]
)
df.groupby("fairness_group")["pm25_mean"].mean()

/var/folders/06/tq96mkbj073500scjzpzwlrw0000gn/T/ipykernel_53439/1505143529.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("fairness_group")["pm25_mean"].mean()


fairness_group
Low         10.587459
Mid-Low      9.836891
Mid-High     9.410766
High         8.527898
Name: pm25_mean, dtype: float64

In [21]:
df.to_csv("Cleaning/final_df_with_index.csv", index=False)

In [74]:
# merge health and smoker dataset and create final_merged_asthma_copd.csv
base_dir = os.getcwd()

cleaning_path = os.path.join(base_dir, "Cleaning")
health_raw_path = os.path.join(base_dir, "Raw", "Health")

final_df = pd.read_csv(os.path.join(cleaning_path, 'final_df_with_index.csv'))
asthma_df = pd.read_csv(os.path.join(health_raw_path, 'Asthma.csv'))
copd_df = pd.read_csv(os.path.join(health_raw_path, 'COPD.csv'))
smoker_df = pd.read_csv(os.path.join(health_raw_path, 'Smoker2020.csv'))

print(f"Project root: {base_dir}")
print(f"Main dataframe loaded with {len(final_df)} rows.")

def clean_prevalence(val):
    if pd.isna(val) or val == '':
        return None
    try:
        clean_str = str(val).replace('%', '').strip()
        return float(clean_str)
    except ValueError:
        return None

asthma_clean = asthma_df[['CensusTract', 'Year', 'Value']].copy()
asthma_clean['asthma_prevalence'] = asthma_clean['Value'].apply(clean_prevalence)
asthma_clean = asthma_clean.drop(columns=['Value']).rename(columns={'CensusTract': 'tract_fips', 'Year': 'year'})
asthma_clean['tract_fips'] = asthma_clean['tract_fips'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(11)

copd_clean = copd_df[['CensusTract', 'Year', 'Value']].copy()
copd_clean['copd_prevalence'] = copd_clean['Value'].apply(clean_prevalence)
copd_clean = copd_clean.drop(columns=['Value']).rename(columns={'CensusTract': 'tract_fips', 'Year': 'year'})
copd_clean['tract_fips'] = copd_clean['tract_fips'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(11)

smoker_clean = smoker_df[['TractFIPS', 'CSMOKING_CrudePrev']].copy()
smoker_clean['smoking_prevalence'] = smoker_clean['CSMOKING_CrudePrev'].apply(clean_prevalence)
smoker_clean = smoker_clean.drop(columns=['CSMOKING_CrudePrev']).rename(columns={'TractFIPS': 'tract_fips'})
smoker_clean['tract_fips'] = smoker_clean['tract_fips'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(11)
smoker_clean['year'] = 2020

final_df['tract_fips'] = final_df['tract_fips'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(11)

df_pre_2020 = final_df[final_df['year'] < 2020].copy()
df_2020_plus = final_df[final_df['year'] >= 2020].copy()

df_pre_2020 = df_pre_2020.merge(asthma_clean, on=['tract_fips', 'year'], how='left')
df_pre_2020 = df_pre_2020.merge(copd_clean, on=['tract_fips', 'year'], how='left')
df_pre_2020['smoking_prevalence'] = None

asthma_2010_tracts = asthma_clean[asthma_clean['year'] == 2020]['tract_fips'].unique()
prefix_2010_map = {str(t)[:9]: t for t in asthma_2010_tracts}

def map_to_2010_fips(t2020):
    s2020 = str(t2020)
    return prefix_2010_map.get(s2020[:9])

df_2020_plus['tract_fips_2010'] = df_2020_plus['tract_fips'].apply(map_to_2010_fips)
df_2020_plus['tract_fips_2010'] = df_2020_plus['tract_fips_2010'].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(11)

asthma_2020 = asthma_clean[asthma_clean['year'] == 2020].rename(columns={'tract_fips': 'tract_fips_2010'})
copd_2020 = copd_clean[copd_clean['year'] == 2020].rename(columns={'tract_fips': 'tract_fips_2010'})
smoker_2020 = smoker_clean.rename(columns={'tract_fips': 'tract_fips_2010'})

df_2020_plus = df_2020_plus.merge(asthma_2020[['tract_fips_2010', 'asthma_prevalence']], on='tract_fips_2010', how='left')
df_2020_plus = df_2020_plus.merge(copd_2020[['tract_fips_2010', 'copd_prevalence']], on='tract_fips_2010', how='left')

df_2020_only = df_2020_plus[df_2020_plus['year'] == 2020].copy()
df_post_2020 = df_2020_plus[df_2020_plus['year'] > 2020].copy()

df_2020_only = df_2020_only.merge(smoker_2020[['tract_fips_2010', 'smoking_prevalence']], on='tract_fips_2010', how='left')
df_post_2020['smoking_prevalence'] = None

df_2020_plus = pd.concat([df_2020_only, df_post_2020], ignore_index=True)
df_2020_plus = df_2020_plus.drop(columns=['tract_fips_2010'])

full_merged = pd.concat([df_pre_2020, df_2020_plus], ignore_index=True)
full_merged.to_csv('Cleaned/final_merged_asthma_copd.csv', index=False)

Project root: /Users/yuttingc1/Downloads/qtmCapstonepollution
Main dataframe loaded with 18508 rows.


In [ ]:
# preparation to merge nasa pm2.5 data and health data
import os
base_dir = os.getcwd()

ZIP_DIR     = os.path.join(base_dir, 'Raw', 'NASAPm2.5Raw')
NC_DIR      = os.path.join(base_dir, 'Cleaning', 'pm25_netcdf_extracted')
SHAPE_2019  = os.path.join(base_dir, 'Raw', 'tl_2019_13_stract', 'tl_2019_13_tract.shp')
SHAPE_2020  = os.path.join(base_dir, 'Raw', 'tl_2020_13_tract', 'tl_2020_13_tract.shp')

CLEAN_DATA  = os.path.join(base_dir, 'Cleaned')

if not os.path.exists(NC_DIR):
    os.makedirs(NC_DIR)

print(f"Project Root: {base_dir}")
print(f"Shapefile 2020 Path: {SHAPE_2020}")

for d in [NC_DIR]:
    os.makedirs(d, exist_ok=True)

GA_LAT_MIN, GA_LAT_MAX = 30.3, 35.1
GA_LON_MIN, GA_LON_MAX = -85.7, -80.7

def find_file(filename):
    for folder in [CLEAN_DATA, '.']:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f'{filename!r} not found in any data folder')



In [35]:
# NASA pm2.5 data cleaning preparation
YEARS = [2018, 2019, 2020, 2021, 2022]

def find_zip_for_year(year, search_dir=ZIP_DIR):
    pattern = re.compile(rf'sdei-global.*-{year}-netcdf.*\.zip$', re.IGNORECASE)
    for fname in os.listdir(search_dir):
        if pattern.match(fname):
            return os.path.join(search_dir, fname)
    return None


def find_nc_for_year(year, search_dir=NC_DIR):
    pattern = re.compile(rf'sdei-global.*-{year}-netcdf\.nc$', re.IGNORECASE)
    for fname in os.listdir(search_dir):
        if pattern.match(fname):
            return os.path.join(search_dir, fname)
    return None


print('Unzipping 5 NASA SDEI files')
for yr in YEARS:
    zip_path = find_zip_for_year(yr)
    if zip_path is None:
        print(f'  {yr}: ⚠ zip not found in {ZIP_DIR}/')
        continue

    if find_nc_for_year(yr) is not None:
        print(f'  {yr}: already extracted ({find_nc_for_year(yr)})')
        continue

    with zipfile.ZipFile(zip_path, 'r') as zf:
        for member in zf.namelist():
            if member.endswith('.nc'):
                zf.extract(member, NC_DIR)
                print(f'  {yr}: extracted {member}')

shp_path = SHAPE_2019 if os.path.exists(SHAPE_2019) else SHAPE_2020
tract_shp = gpd.read_file(shp_path).to_crs(epsg=4326)
tract_shp['GEOID'] = tract_shp['GEOID'].astype(str).str.zfill(11)
print(f'Loaded shapefile: {shp_path}')
print(f'Georgia census tracts: {len(tract_shp)}')

Unzipping 5 NASA SDEI files
  2018: extracted sdei-global-annual-gwr-pm2-5-modis-misr-seawifs-viirs-aod-v5-gl-04-2018-netcdf.nc
  2019: extracted sdei-global-annual-gwr-pm2-5-modis-misr-seawifs-viirs-aod-v5-gl-04-2019-netcdf.nc
  2020: extracted sdei-global-annual-gwr-pm2-5-modis-misr-seawifs-viirs-aod-v5-gl-04-2020-netcdf.nc
  2021: extracted sdei-global-annual-gwr-pm2-5-modis-misr-seawifs-viirs-aod-v5-gl-04-2021-netcdf.nc
  2022: extracted sdei-global-annual-gwr-pm2-5-modis-misr-seawifs-viirs-aod-v5-gl-04-2022-netcdf.nc
Loaded shapefile: Raw/tl_2020_13_tract/tl_2020_13_tract.shp
Georgia census tracts: 2796


In [62]:
# nasa data cleaning
YEARS = [2018, 2019, 2020, 2021, 2022]
ZIP_DIR, NC_DIR, CLEANING, CLEAN_DATA = 'Raw/NASAPm2.5Raw', 'Cleaning/pm25_netcdf_extracted', 'Cleaning', 'Cleaned/'
SHAPE_2019  = 'Raw/tl_2019_13_tract/tl_2019_13_tract.shp'
SHAPE_2020  = 'Raw/tl_2020_13_tract/tl_2020_13_tract.shp'
GA_LAT = (30.3, 35.1)
GA_LON = (-85.7, -80.7)

os.makedirs(CLEAN_DATA, exist_ok=True)
os.makedirs(NC_DIR, exist_ok=True)

def prep_shp(path):
    gdf = gpd.read_file(path)
    gdf = gdf.to_crs(epsg=4326) 
    gdf['GEOID'] = (gdf['GEOID'].astype(str)
                                .str.replace(r'\.0$', '', regex=True)
                                .str.zfill(11))
    return gdf
    
tract_2019 = prep_shp(SHAPE_2019)
tract_2020 = prep_shp(SHAPE_2020)

yearly_dfs = []

for yr in YEARS:

    current_shp = tract_2019 if yr <= 2019 else tract_2020
    

    zip_fn = next((f for f in os.listdir(ZIP_DIR) if f"sdei-global-annual-gwr-pm2-5-modis-misr-seawifs-v4-gl-03-{yr}" in f.lower()), None)
    if zip_fn:
        with zipfile.ZipFile(os.path.join(ZIP_DIR, zip_fn), 'r') as zf:
            for m in zf.namelist():
                if m.endswith('.nc'): zf.extract(m, NC_DIR)


    nc_file = next((os.path.join(NC_DIR, f) for f in os.listdir(NC_DIR) if str(yr) in f and f.endswith('.nc')), None)
    if nc_file:
        ds = xr.open_dataset(nc_file)
        da = ds['GWRPM25']
        lat_slice = slice(GA_LAT[0], GA_LAT[1]) if ds.lat.values[0] < ds.lat.values[-1] else slice(GA_LAT[1], GA_LAT[0])
        ga_raster = da.sel(lat=lat_slice, lon=slice(GA_LON[0], GA_LON[1]))
        

        arr = ga_raster.values.astype(np.float64)
        lats, lons = ga_raster['lat'].values, ga_raster['lon'].values
        if lats[0] > lats[-1]:
            arr, lats = np.flipud(arr), lats[::-1]
            
        affine = from_bounds(lons.min(), lats.min(), lons.max(), lats.max(), arr.shape[1], arr.shape[0])
        stats = zonal_stats(current_shp.geometry, arr, affine=affine, stats='mean', nodata=np.nan, all_touched=True)
        
    
        year_df = pd.DataFrame({
            'GEOID': current_shp['GEOID'],
            f'PM25_{yr}': [s['mean'] for s in stats]
        })
        yearly_dfs.append(year_df)

results = reduce(lambda left, right: pd.merge(left, right, on='GEOID', how='outer'), yearly_dfs)

yr_cols = [c for c in results.columns if 'PM25_' in c]
results['PM25_mean_2018_2022'] = results[yr_cols].mean(axis=1)

results.to_csv(os.path.join(CLEANING, 'pm25_tract_2018_2022.csv'), index=False)

In [63]:
# merge nasa pm2.5 with cdc health data
def clean_cdc_health(path, col_name):
    """
    Cleans Asthma or COPD CSVs. Detects FIPS and Value columns,
    normalizes GEOID to 11 digits, and converts values to numeric.
    """
    df = pd.read_csv(path)
    
    t_col = next(c for c in ['CensusTract', 'TractFIPS', 'GEOID', 'GeoID'] if c in df.columns)
    v_col = next(c for c in ['Value', 'Data_Value', 'Crude_Prevalence'] if c in df.columns)
    
    df['GEOID'] = df[t_col].astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(11)
    df[col_name] = pd.to_numeric(df[v_col].astype(str).str.replace('%', ''), errors='coerce')
    
    return df[['GEOID', col_name]].dropna().drop_duplicates(subset='GEOID')

asthma = clean_cdc_health('Raw/Health/Asthma.csv', 'Asthma_Prev')
copd = clean_cdc_health('Raw/Health/COPD.csv', 'COPD_Prev')

import censusdata

ga_acs = censusdata.download('acs5', 2022, 
    censusdata.censusgeo([('state', '13'), ('county', '*'), ('tract', '*')]),
    ['B17001_001E', 'B17001_002E', 'B01003_001E', 'B02001_003E', 'B15003_022E', 'B15003_001E'])

ga_acs = ga_acs.reset_index()

ga_acs['GEOID'] = ga_acs['index'].apply(lambda x: "".join([v.zfill(i) for v, i in zip(dict(x.params()).values(), [2,3,6])]))

ga_acs['Poverty_Rate'] = (ga_acs['B17001_002E'] / ga_acs['B17001_001E'] * 100).clip(0, 100)
ga_acs['Pct_Black'] = (ga_acs['B02001_003E'] / ga_acs['B01003_001E'] * 100).clip(0, 100)
ga_acs['Pct_College'] = (ga_acs['B15003_022E'] / ga_acs['B15003_001E'] * 100).clip(0, 100)

acs_final = ga_acs[['GEOID', 'Poverty_Rate', 'Pct_Black', 'Pct_College']].dropna()

final_merge = results.merge(copd, on='GEOID', how='left') \
                     .merge(asthma, on='GEOID', how='left') \
                     .merge(acs_final, on='GEOID', how='left')

final_merge = final_merge.dropna(subset=['Asthma_Prev', 'COPD_Prev'])

output_path = os.path.join(CLEAN_DATA, 'merged_regression_ready.csv')
final_merge.to_csv(output_path, index=False)

print(f"Total tracts included: {len(final_merge)}")

Total tracts included: 3492


In [64]:
# final data summary for merged_regression_ready.csv
final_csv_path = os.path.join('Cleaned', 'merged_regression_ready.csv')

if os.path.exists(final_csv_path):
    merged_final = pd.read_csv(final_csv_path)
    
    print('Final dataset summary')
    print(f'  Rows (unique tracts): {len(merged_final)}')
    print()
    
    print('  Variable ranges:')
    cols_to_check = [
        'PM25_mean_2018_2022', 'COPD_Prev', 'Asthma_Prev',
        'Poverty_Rate', 'Pct_Black', 'Pct_College'
    ]
    
    for col in cols_to_check:
        if col in merged_final.columns:
            vals = merged_final[col].dropna()
            print(f'    {col:<22} min={vals.min():>6.2f}  median={vals.median():>6.2f}  max={vals.max():>6.2f}')
        else:
            print(f'    ⚠ Column {col} not found in dataset.')

    print('\n  Quick correlations vs COPD:')
    analysis_vars = ['PM25_mean_2018_2022', 'Poverty_Rate', 'Pct_Black', 'Pct_College']
    
    for col in analysis_vars:
        if col in merged_final.columns and 'COPD_Prev' in merged_final.columns:
            df_v = merged_final.dropna(subset=[col, 'COPD_Prev'])
            
            if len(df_v) > 1:
                r, p = pearsonr(df_v[col], df_v['COPD_Prev'])
                print(f'    {col:<22} r = {r:+.3f}, p = {p:.4f}')
            else:
                print(f'    {col:<22} Not enough valid data for correlation.')
        else:
            print(f'    {col:<22} Missing necessary columns for correlation.')

else:
    print(f"Error: Could not find the file at {final_csv_path}. Please check your 'Cleaned' folder.")

Final dataset summary
  Rows (unique tracts): 3492

  Variable ranges:
    PM25_mean_2018_2022    min=  5.39  median=  7.01  max=  9.10
    COPD_Prev              min=  1.60  median=  7.10  max= 17.90
    Asthma_Prev            min=  6.20  median= 10.00  max= 15.80
    Poverty_Rate           min=  0.00  median= 12.05  max= 84.68
    Pct_Black              min=  0.00  median= 24.10  max=100.00
    Pct_College            min=  0.00  median= 16.89  max= 63.83

  Quick correlations vs COPD:
    PM25_mean_2018_2022    r = -0.047, p = 0.0050
    Poverty_Rate           r = +0.555, p = 0.0000
    Pct_Black              r = +0.142, p = 0.0000
    Pct_College            r = -0.747, p = 0.0000
